# GeoForest — Roboflow Export Notebook
**Kaggle Sentinel-2 Ghana → PNG chips → Roboflow project**

This notebook is a **standalone export pipeline**. It does one job:
take your raw Sentinel-2 GeoTIFF tiles, convert them to annotator-friendly
PNG chips, and push everything to Roboflow so you can do clean manual
annotations there.

**What this notebook produces:**
- `rgb/`          — standard RGB PNGs (what Roboflow displays by default)
- `falsecolour/`  — NIR-Red-Green false-colour PNGs (makes forest/bare-soil pop)
- `ndvi_preview/` — NDVI colourmap PNGs (RdYlGn, for annotation reference)
- `annotations.json` — NDVI pseudo-label COCO JSON pre-loaded as annotation hints
- All images + labels uploaded to your Roboflow project via the API

**Workflow:**
1. Run all cells top-to-bottom once
2. Open your Roboflow project → Annotate
3. Pseudo-label boxes are already there as a starting point — correct/add/delete
4. Export from Roboflow as COCO JSON → use in the training notebook

> **This notebook does NOT train anything.** It is purely for data prep and upload.


## 0 — Install dependencies

In [ ]:
%%capture
!pip install rasterio roboflow pillow tqdm -q


## 1 — Imports

In [ ]:
import os, json, shutil, warnings
from pathlib import Path

import numpy as np
import rasterio
from rasterio.enums import Resampling
from PIL import Image
import matplotlib
matplotlib.use('Agg')          # headless — no display needed in this notebook
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
print('✓ imports ok')


## 2 — Configuration

**Fill in your Roboflow credentials here before running.**

Get your API key from: https://app.roboflow.com → Settings → Roboflow API


In [ ]:
# ── Dataset paths ─────────────────────────────────────────────────────────────
DATASET_ROOT = Path('/kaggle/input/datasets/kwabenaobeng/ghana-sentinel2-forest')
GEOTIFF_DIR  = DATASET_ROOT

OUTPUT_DIR   = Path('/kaggle/working/roboflow_export')
RGB_DIR      = OUTPUT_DIR / 'rgb'
FC_DIR       = OUTPUT_DIR / 'falsecolour'
NDVI_DIR     = OUTPUT_DIR / 'ndvi_preview'
COCO_PATH    = OUTPUT_DIR / 'annotations.json'

for d in [OUTPUT_DIR, RGB_DIR, FC_DIR, NDVI_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Image export settings ─────────────────────────────────────────────────────
CHIP_SIZE  = 512      # px — all exports are this resolution
JPEG_Q     = 92       # JPEG quality for RGB / false-colour (smaller files)
NDVI_DPI   = 120      # DPI for NDVI colourmap PNGs

# ── NDVI pseudo-label thresholds (used to pre-populate annotation hints) ──────
NDVI_FOREST_THRESH   = 0.45   # NDVI ≥ this → forest box
NDVI_DEGRADED_THRESH = 0.20   # NDVI <  this → degraded box
MIN_BLOB_AREA_PX     = 80     # minimum blob area in the 512² tile
MAX_BOXES_PER_CLASS  = 40
MAX_BOX_AREA_FRAC    = 0.12

# Cloud-masking (keeps clouds out of annotation boxes)
CLOUD_BRIGHTNESS_THRESH  = 0.35
SHADOW_BRIGHTNESS_THRESH = 0.02
CLOUD_DILATE_PX          = 4

# ── Roboflow credentials ──────────────────────────────────────────────────────
# 1) Get your API key from: https://app.roboflow.com → Settings → Roboflow API
# 2) RF_WORKSPACE is auto-resolved from your key — leave as "" or any placeholder,
#    cell 16 will print the correct slug and use it.
# 3) RF_PROJECT must be a URL-SAFE SLUG: lowercase, hyphens/digits OK, NO spaces,
#    NO underscores. Examples: "ghana-forest-cover", "s2-forest-ghana".
RF_API_KEY        = "YOUR_API_KEY_HERE"       # ← paste your key
RF_WORKSPACE      = ""                         # leave blank — auto-resolved
RF_PROJECT        = "ghana-forest-cover"      # URL-safe slug (letters + digits + hyphens)
RF_PROJECT_NAME   = "Ghana Forest Cover"      # display name (spaces are fine)
RF_PROJECT_TYPE   = "object-detection"        # do not change
RF_ANNOTATION_GROUP = "forestcover"           # ALPHANUMERIC ONLY — no hyphens/underscores/spaces
RF_PROJECT_LICENSE  = "Public Domain"         # free-plan safe; use "Private" only on paid plans
RF_BATCH_NAME       = "sentinel2-batch-1"     # label for this upload batch

# ── Upload switches ───────────────────────────────────────────────────────────
UPLOAD_RGB         = True    # upload RGB images (required)
UPLOAD_FALSECOLOUR = False   # upload false-colour as a second batch (optional)
UPLOAD_ANNOTATIONS = True    # pre-load NDVI pseudo-label boxes as hints

tif_files = sorted(GEOTIFF_DIR.glob('*.tif'))
print(f'Found {len(tif_files)} GeoTIFF file(s)')
for f in tif_files[:6]:
    print(f'  {f.name}')
if len(tif_files) > 6:
    print(f'  ... and {len(tif_files)-6} more')


## 3 — Core image helpers

NaN-aware read, cloud masking, NDVI, and three render modes:
- **RGB** — display-ready, percentile-stretched
- **False colour** — NIR → R, Red → G, Green → B (standard vegetation false colour)
- **NDVI colourmap** — RdYlGn matplotlib colourmap saved as PNG for reference


In [ ]:
from scipy import ndimage as ndi
from skimage import measure, morphology


# ─────────────────────────────────────────────────────────────────────────────
# I/O
# ─────────────────────────────────────────────────────────────────────────────

def read_tif(path, target_size=CHIP_SIZE):
    """Read GeoTIFF → (C, H, W) float32, NaN for nodata/±inf, resampled to target_size."""
    with rasterio.open(path) as src:
        nodata = src.nodata
        bands  = src.read(
            out_shape=(src.count, target_size, target_size),
            resampling=Resampling.lanczos,
            out_dtype='float32',
        )
    if nodata is not None:
        bands[bands == nodata] = np.nan
    bands = np.where(np.isinf(bands), np.nan, bands)
    return bands


# ─────────────────────────────────────────────────────────────────────────────
# Cloud / shadow masking
# ─────────────────────────────────────────────────────────────────────────────

def detect_clouds(bands):
    """
    Heuristic cloud+shadow mask. Returns (H, W) bool — True = masked.
    Works without an SCL band: uses brightness + NDVI heuristics.
    """
    vmax = float(np.nanmax(np.abs(bands))) if not np.all(np.isnan(bands)) else 1.0
    norm = 10000.0 if vmax > 10.0 else 1.0
    b    = np.clip(bands / norm, 0, 1)

    brightness = np.nanmean(b[:4], axis=0)
    red_s = np.nan_to_num(b[2], nan=0.0)
    nir_s = np.nan_to_num(b[3], nan=0.0)
    denom = nir_s + red_s
    ndvi_q = np.where(denom == 0, 0.0, (nir_s - red_s) / denom)

    cloud  = (brightness > CLOUD_BRIGHTNESS_THRESH) & (ndvi_q < 0.25)
    shadow = brightness < SHADOW_BRIGHTNESS_THRESH
    mask   = cloud | shadow

    if CLOUD_DILATE_PX > 0:
        struct = ndi.generate_binary_structure(2, 1)
        mask   = ndi.binary_dilation(mask, structure=struct, iterations=CLOUD_DILATE_PX)

    mask = mask | np.any(np.isnan(bands), axis=0)
    return mask.astype(bool)


# ─────────────────────────────────────────────────────────────────────────────
# NDVI
# ─────────────────────────────────────────────────────────────────────────────

def compute_ndvi(bands):
    """(C,H,W) → (H,W) float32 in [-1,1], NaN where red/NIR invalid."""
    red, nir = bands[2].astype('float32'), bands[3].astype('float32')
    invalid  = np.isnan(red) | np.isnan(nir)
    denom    = np.nan_to_num(nir, nan=0.) + np.nan_to_num(red, nan=0.)
    ndvi     = np.where(invalid | (denom == 0), np.nan,
                        (np.nan_to_num(nir, nan=0.) - np.nan_to_num(red, nan=0.)) / denom)
    return np.clip(ndvi, -1., 1.).astype('float32')


# ─────────────────────────────────────────────────────────────────────────────
# Normalisation
# ─────────────────────────────────────────────────────────────────────────────

def pct_norm(arr, lo=2, hi=98):
    """Percentile-stretch to [0,1]. NaN excluded then zeroed."""
    valid = arr[~np.isnan(arr)]
    if valid.size == 0:
        return np.zeros_like(arr)
    p_lo, p_hi = np.percentile(valid, [lo, hi])
    return np.nan_to_num((arr - p_lo) / (p_hi - p_lo + 1e-8), nan=0.).clip(0, 1)


# ─────────────────────────────────────────────────────────────────────────────
# Three render modes
# ─────────────────────────────────────────────────────────────────────────────

def render_rgb(bands):
    """Standard RGB — channel order B=0 G=1 R=2. Returns uint8 (H,W,3)."""
    r = pct_norm(bands[2])
    g = pct_norm(bands[1])
    b = pct_norm(bands[0])
    return (np.stack([r, g, b], axis=-1) * 255).astype('uint8')


def render_falsecolour(bands):
    """
    NIR-Red-Green false colour — standard vegetation composite.
    Dense forest → bright red/magenta. Bare soil / degraded → cyan/blue-green.
    Returns uint8 (H,W,3).
    """
    r = pct_norm(bands[3])   # NIR   → Red channel
    g = pct_norm(bands[2])   # Red   → Green channel
    b = pct_norm(bands[1])   # Green → Blue channel
    return (np.stack([r, g, b], axis=-1) * 255).astype('uint8')


def render_ndvi_map(ndvi, dpi=NDVI_DPI):
    """
    Save NDVI as a RdYlGn colourmap PNG using matplotlib.
    Returns a PIL Image (RGBA uint8).
    """
    H, W  = ndvi.shape
    fig   = plt.figure(figsize=(W / dpi, H / dpi), dpi=dpi)
    ax    = fig.add_axes([0, 0, 1, 1])
    ax.imshow(ndvi, cmap='RdYlGn', vmin=-0.2, vmax=0.8, interpolation='nearest')
    ax.axis('off')

    fig.canvas.draw()
    buf = fig.canvas.buffer_rgba()
    img = Image.frombytes('RGBA', fig.canvas.get_width_height(), buf)
    plt.close(fig)
    return img.convert('RGB')


print('✓ image helpers defined')


## 4 — Export PNG chips

Generates three sets of PNGs for every tile:
- **RGB** — what Roboflow shows annotators by default
- **False colour** — NIR-R-G, makes forest vs bare soil instantly visible
- **NDVI preview** — colour-mapped NDVI, useful as a side-reference during annotation

All saved at `CHIP_SIZE × CHIP_SIZE` px.


In [ ]:
def export_all_pngs(tif_files):
    """
    Export RGB, false-colour and NDVI preview PNGs for every tile.
    Skips files that already exist (safe to re-run).
    """
    skipped = done = 0

    for path in tqdm(tif_files, desc='Exporting PNGs'):
        stem = path.stem

        rgb_out  = RGB_DIR  / f'{stem}.jpg'
        fc_out   = FC_DIR   / f'{stem}.jpg'
        ndvi_out = NDVI_DIR / f'{stem}.png'

        all_exist = rgb_out.exists() and fc_out.exists() and ndvi_out.exists()
        if all_exist:
            skipped += 1
            continue

        bands = read_tif(path)
        ndvi  = compute_ndvi(bands)

        if not rgb_out.exists():
            Image.fromarray(render_rgb(bands)).save(rgb_out, quality=JPEG_Q)

        if not fc_out.exists():
            Image.fromarray(render_falsecolour(bands)).save(fc_out, quality=JPEG_Q)

        if not ndvi_out.exists():
            render_ndvi_map(ndvi).save(ndvi_out)

        done += 1

    print(f'✓ Exported {done} tile(s)  |  {skipped} already existed')
    print(f'  RGB         → {RGB_DIR}')
    print(f'  False colour → {FC_DIR}')
    print(f'  NDVI preview → {NDVI_DIR}')


export_all_pngs(tif_files)


## 5 — Quick visual check

Shows one tile in all three render modes side by side.
Run this to confirm exports look correct before uploading.


In [ ]:
import matplotlib
matplotlib.use('inline')   # switch back to inline for display
%matplotlib inline

def preview_exports(stem):
    rgb_path  = RGB_DIR  / f'{stem}.jpg'
    fc_path   = FC_DIR   / f'{stem}.jpg'
    ndvi_path = NDVI_DIR / f'{stem}.png'

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    fig.suptitle(stem, fontsize=11, y=1.01)

    for ax, p, title in zip(axes,
                             [rgb_path, fc_path, ndvi_path],
                             ['RGB (annotator view)',
                              'False colour NIR-R-G',
                              'NDVI preview (RdYlGn)']):
        if p.exists():
            ax.imshow(Image.open(p))
            ax.set_title(title, fontsize=10)
        else:
            ax.set_title(f'{title}\n(not found)', fontsize=10, color='red')
        ax.axis('off')

    plt.tight_layout()
    plt.show()


# Preview the first tile that was exported
exported = sorted(RGB_DIR.glob('*.jpg'))
if exported:
    preview_exports(exported[0].stem)
else:
    print('No exports found — run cell 4 first.')


## 6 — Generate NDVI pseudo-label COCO annotations

These are **not your final annotations** — they are starting-point bounding boxes
generated from NDVI thresholds. When uploaded to Roboflow they appear as pre-drawn
boxes that you can correct, add to, or delete.

This is much faster than drawing every box from scratch.


In [ ]:
def ndvi_to_boxes(ndvi, cloud_mask, label_id, mask_fn):
    """Connected-component labelling on cloud-free NDVI mask. Returns COCO boxes."""
    ndvi_safe = np.nan_to_num(ndvi, nan=0.0)
    mask      = mask_fn(ndvi_safe).astype('uint8')
    mask[cloud_mask] = 0

    if mask.sum() == 0:
        return []

    mask = morphology.binary_closing(mask, footprint=morphology.disk(3)).astype('uint8')
    mask = morphology.remove_small_objects(mask.astype(bool),
                                           min_size=MIN_BLOB_AREA_PX).astype('uint8')

    labels_map = measure.label(mask, connectivity=2)
    props      = measure.regionprops(labels_map)
    tile_area  = ndvi.shape[0] * ndvi.shape[1]

    props = [p for p in props
             if p.area >= MIN_BLOB_AREA_PX
             and p.area < MAX_BOX_AREA_FRAC * tile_area]
    props = sorted(props, key=lambda p: p.area, reverse=True)[:MAX_BOXES_PER_CLASS]

    boxes = []
    for p in props:
        min_r, min_c, max_r, max_c = p.bbox
        x, y = int(min_c), int(min_r)
        w, h = int(max_c - min_c), int(max_r - min_r)
        if w > 0 and h > 0:
            boxes.append({
                'bbox':        [x, y, w, h],
                'category_id': label_id,
                'area':        float(p.area),
                'iscrowd':     0,
            })
    return boxes


def build_coco(tif_files, output_json):
    """
    Build a COCO annotations.json from NDVI pseudo-labels.
    file_name fields use .jpg so they match the RGB exports uploaded to Roboflow.
    """
    coco = {
        'info': {
            'description': 'GeoForest Ghana — NDVI pseudo-labels for Roboflow',
            'version': '1.0',
            'tile_size': CHIP_SIZE,
        },
        'categories': [
            {'id': 0, 'name': 'forest',   'supercategory': 'vegetation'},
            {'id': 1, 'name': 'degraded', 'supercategory': 'vegetation'},
        ],
        'images':      [],
        'annotations': [],
    }

    ann_id = 0

    for img_id, path in enumerate(tqdm(tif_files, desc='Building COCO labels')):
        bands    = read_tif(path)
        cmask    = detect_clouds(bands)
        cfrac    = float(cmask.mean())
        ndvi     = compute_ndvi(bands)

        ndvi_mean = float(np.nanmean(ndvi[~cmask])) if (~cmask).any() else 0.0
        ndvi_std  = float(np.nanstd(ndvi[~cmask]))  if (~cmask).any() else 0.0

        # Use .jpg extension to match the RGB files Roboflow will receive
        jpg_name = Path(path.name).with_suffix('.jpg').name

        coco['images'].append({
            'id':          img_id,
            'file_name':   jpg_name,
            'width':       CHIP_SIZE,
            'height':      CHIP_SIZE,
            'ndvi_mean':   ndvi_mean,
            'ndvi_std':    ndvi_std,
            'cloud_frac':  cfrac,
        })

        if cfrac > 0.70:
            print(f'  ⚠ {path.name} — {100*cfrac:.0f} % cloud/nodata, skipping labels')
            continue

        for box in ndvi_to_boxes(ndvi, cmask, 0, lambda n: n >= NDVI_FOREST_THRESH):
            box.update({'id': ann_id, 'image_id': img_id})
            coco['annotations'].append(box)
            ann_id += 1

        for box in ndvi_to_boxes(ndvi, cmask, 1, lambda n: n < NDVI_DEGRADED_THRESH):
            box.update({'id': ann_id, 'image_id': img_id})
            coco['annotations'].append(box)
            ann_id += 1

    with open(output_json, 'w') as f:
        json.dump(coco, f, indent=2)

    n_imgs  = len(coco['images'])
    n_anns  = len(coco['annotations'])
    n_for   = sum(1 for a in coco['annotations'] if a['category_id'] == 0)
    n_deg   = n_anns - n_for

    print(f'\n✓ COCO annotations saved → {output_json}')
    print(f'  Images      : {n_imgs}')
    print(f'  Total boxes : {n_anns}  (forest={n_for}, degraded={n_deg})')

    # Per-tile summary
    counts = {img['id']: 0 for img in coco['images']}
    for ann in coco['annotations']:
        counts[ann['image_id']] += 1
    print('\nPer-tile box counts:')
    for img in coco['images']:
        n = counts[img['id']]
        flag = '  ← 0 boxes (check cloud mask or thresholds)' if n == 0 else ''
        print(f'  {img["file_name"]:40s}  {n:3d} boxes  '
              f'cloud={100*img["cloud_frac"]:.0f}%{flag}')

    return coco


coco_data = build_coco(tif_files, COCO_PATH)


## 7 — Connect to Roboflow

Creates (or opens) your project. Run this once.
Make sure `RF_API_KEY`, `RF_WORKSPACE`, and `RF_PROJECT` are set in cell 2.


In [ ]:
from roboflow import Roboflow

if RF_API_KEY == "YOUR_API_KEY_HERE":
    raise ValueError(
        "Paste your Roboflow API key into RF_API_KEY in cell 6 before running this cell.\n"
        "Get it from: https://app.roboflow.com → Settings → Roboflow API"
    )

rf = Roboflow(api_key=RF_API_KEY)

# ── Step 1: resolve workspace from API key (no slug arg needed) ───────────────
# rf.workspace() auto-resolves the workspace tied to the API key. The slug it
# prints here is the one Roboflow actually uses — ignore whatever was in
# RF_WORKSPACE at the top of the notebook.
workspace = rf.workspace()
actual_slug = workspace.url
RF_WORKSPACE = actual_slug
print(f'Workspace slug (auto-resolved): "{actual_slug}"')


# ── Step 2: list existing projects so we can detect already-created ones ──────
# The Roboflow API returns "does not exist" both for projects that truly don\'t
# exist AND for projects created under a slightly different slug (e.g. you
# created "Ghana Forest Cover" manually in the UI and Roboflow gave it the slug
# "ghana-forest-cover-abc1").  Listing the workspace projects first lets us
# recover from that gracefully instead of hitting the create endpoint again.
def list_existing_projects(ws):
    """Return a list of project slugs in this workspace (best-effort)."""
    try:
        projs = ws.projects()               # newer SDK versions
        slugs = []
        for p in projs:
            # Each item is either a Project obj or a dict; handle both.
            if hasattr(p, 'id'):
                slugs.append(p.id.split('/')[-1])
            elif isinstance(p, dict):
                slugs.append(p.get('id', '').split('/')[-1])
            else:
                slugs.append(str(p).split('/')[-1])
        return [s for s in slugs if s]
    except Exception:
        pass
    # Fallback: raw API call
    try:
        import requests
        url = f'https://api.roboflow.com/{ws.url}?api_key={RF_API_KEY}'
        r = requests.get(url, timeout=15)
        if r.ok:
            data = r.json()
            wk  = data.get('workspace', {})
            return [p.get('id', '').split('/')[-1]
                    for p in wk.get('projects', [])]
    except Exception:
        pass
    return []


existing = list_existing_projects(workspace)
if existing:
    print(f'Existing projects in this workspace: {existing}')


# ── Step 3: open or create the project ────────────────────────────────────────
project = None

# (a) Try the configured slug directly.
try:
    project = workspace.project(RF_PROJECT)
    print(f'✓ Opened existing project: {RF_PROJECT}')
except Exception as open_err:
    err_str = str(open_err).lower()
    is_missing = any(k in err_str for k in
                     ('does not exist', 'cannot be loaded',
                      'graphmethod', '404', 'permission'))
    if not is_missing:
        print(f'✗ Unexpected error opening project: {open_err}')
        raise

# (b) If configured slug didn\'t work but something SIMILAR exists, open that.
if project is None and existing:
    target = RF_PROJECT.lower().replace('_', '-')
    matches = [s for s in existing
               if s == target
               or s.startswith(target + '-')
               or target.startswith(s)
               or s.replace('-', '') == target.replace('-', '')]
    if matches:
        RF_PROJECT = matches[0]
        print(f'↪ Found similar project slug "{RF_PROJECT}" — opening it instead')
        try:
            project = workspace.project(RF_PROJECT)
            print(f'✓ Opened existing project: {RF_PROJECT}')
        except Exception as e:
            print(f'✗ Could not open matched project {RF_PROJECT}: {e}')
            project = None

# (c) Still nothing — create a fresh project.
if project is None:
    print(f'Project "{RF_PROJECT}" not found — creating it...')
    print(f'  name        : {RF_PROJECT_NAME}')
    print(f'  type        : {RF_PROJECT_TYPE}')
    print(f'  license     : {RF_PROJECT_LICENSE}')
    print(f'  annotation  : {RF_ANNOTATION_GROUP}  (must be alphanumeric only)')

    # Validation — catches the two most common 422 causes BEFORE the API call.
    if not RF_ANNOTATION_GROUP.isalnum():
        raise ValueError(
            f'RF_ANNOTATION_GROUP="{RF_ANNOTATION_GROUP}" contains non-alphanumeric '
            'characters. The Roboflow API requires letters/digits only — no hyphens, '
            'underscores, or spaces. Fix it in cell 6 and re-run.'
        )
    if RF_PROJECT_LICENSE not in {
        "Public Domain", "MIT", "CC BY 4.0", "BY-NC-SA 4.0", "OBdL v1.0", "Private"
    }:
        print(f'  ⚠ RF_PROJECT_LICENSE="{RF_PROJECT_LICENSE}" is not one of the '
              f'accepted values — the API may return 422.')

    try:
        project = workspace.create_project(
            project_name    = RF_PROJECT_NAME,
            project_type    = RF_PROJECT_TYPE,
            project_license = RF_PROJECT_LICENSE,
            annotation      = RF_ANNOTATION_GROUP,
        )
        print(f'✓ Created project')

        # Roboflow generates the real URL slug from project_name.  If it
        # differs from what RF_PROJECT was set to, update RF_PROJECT so the
        # rest of the notebook uses the right value.
        assigned_slug = None
        if hasattr(project, 'id'):
            assigned_slug = project.id.split('/')[-1]
        if assigned_slug and assigned_slug != RF_PROJECT:
            print(f'  Roboflow assigned slug: "{assigned_slug}" '
                  f'(differs from RF_PROJECT="{RF_PROJECT}")')
            print(f'  Using "{assigned_slug}" from here on.')
            RF_PROJECT = assigned_slug

    except Exception as create_err:
        print(f'\n✗ create_project failed: {create_err}\n')
        print('Most common causes of a 422 here:')
        print('  1. project_license="Private" on a FREE plan')
        print('     → fix: use "Public Domain" or "MIT" in RF_PROJECT_LICENSE')
        print('  2. annotation field has non-alphanumeric chars')
        print('     → fix: RF_ANNOTATION_GROUP must be letters+digits only')
        print('  3. Free-plan project limit reached (3 projects)')
        print('     → fix: delete an old project at https://app.roboflow.com')
        print('  4. A project with the same name already exists but under')
        print('     a different slug — this cell should have caught it above,')
        print('     but you can check at https://app.roboflow.com')
        print('\nFastest workaround:')
        print('  - Create the project manually in the Roboflow UI')
        print(f'  - Copy its URL slug into RF_PROJECT in cell 6')
        print('  - Re-run this cell (it will just open the existing project)')
        raise

print()
print(f'  Workspace   : {RF_WORKSPACE}')
print(f'  Project     : {RF_PROJECT}')
print(f'  Type        : {RF_PROJECT_TYPE}')
print(f'  Project URL : https://app.roboflow.com/{RF_WORKSPACE}/{RF_PROJECT}')


## 8 — Verify annotation classes

Roboflow creates classes when annotations are first uploaded.
This cell prints a reminder of the two class names so you know
what to expect in the labelling UI.


In [ ]:
CLASS_MAP = {
    0: 'forest',
    1: 'degraded',
}

print('Classes that will appear in Roboflow:')
for cid, name in CLASS_MAP.items():
    n = sum(1 for a in coco_data['annotations'] if a['category_id'] == cid)
    print(f'  [{cid}] {name:12s} — {n} pseudo-label boxes')

print()
print('These match the category names in annotations.json.')
print('Roboflow will create them automatically on first upload.')


## 9 — Split COCO into per-image annotation files

Roboflow's upload API accepts one annotation file per image.
This cell splits the single `annotations.json` into individual
files in `OUTPUT_DIR/per_image_anns/` — one JSON per tile.

The format used is **Roboflow-compatible JSON** (a simple list of boxes).


In [ ]:
PER_IMAGE_DIR = OUTPUT_DIR / 'per_image_anns'
PER_IMAGE_DIR.mkdir(exist_ok=True)

# Build lookup: image_id → image meta
id_to_meta = {img['id']: img for img in coco_data['images']}
# Build lookup: image_id → list of annotations
id_to_anns = {img['id']: [] for img in coco_data['images']}
for ann in coco_data['annotations']:
    id_to_anns[ann['image_id']].append(ann)

written = 0
for img in coco_data['images']:
    iid     = img['id']
    anns    = id_to_anns[iid]
    stem    = Path(img['file_name']).stem

    # Roboflow annotation JSON format: list of {x,y,width,height,label}
    # x/y are the box CENTRE normalised to [0,1], width/height normalised to [0,1]
    W, H = img['width'], img['height']
    rf_anns = []
    for ann in anns:
        bx, by, bw, bh = ann['bbox']   # COCO: top-left x,y + w,h in pixels
        cx = (bx + bw / 2) / W
        cy = (by + bh / 2) / H
        nw = bw / W
        nh = bh / H
        rf_anns.append({
            'x':      round(cx, 6),
            'y':      round(cy, 6),
            'width':  round(nw, 6),
            'height': round(nh, 6),
            'label':  CLASS_MAP[ann['category_id']],
        })

    out_path = PER_IMAGE_DIR / f'{stem}.json'
    with open(out_path, 'w') as f:
        json.dump(rf_anns, f, indent=2)
    written += 1

print(f'✓ Wrote {written} per-image annotation files → {PER_IMAGE_DIR}')
total_boxes = len(coco_data['annotations'])
print(f'  Total pseudo-label boxes across all files: {total_boxes}')


## 10 — Upload to Roboflow

Uploads every RGB image together with its pseudo-label annotation file.

**What each image gets:**
- The `.jpg` RGB chip — what you see in the Roboflow annotator
- The per-image `.json` annotation — pre-drawn boxes you can accept/edit/delete

Set `UPLOAD_FALSECOLOUR = True` in cell 2 to also push the false-colour
images as a second batch (useful as visual reference during annotation).

The upload is resumable — already-uploaded images are skipped if you re-run.


In [ ]:
def upload_batch(project, image_files, ann_dir=None, batch_name=RF_BATCH_NAME):
    """
    Upload a list of image files to the Roboflow project.
    If ann_dir is provided, looks for a matching .json annotation file.
    Handles SDK-version differences in upload() kwargs.
    """
    uploaded = errors = skipped = 0

    for img_path in tqdm(image_files, desc=f'Uploading to {RF_PROJECT}'):
        ann_path = None
        if ann_dir:
            candidate = ann_dir / f'{img_path.stem}.json'
            if candidate.exists():
                ann_path = str(candidate)

        # Build kwargs defensively — older roboflow packages don\'t accept
        # is_prediction or tag_names on .upload().
        kwargs = {
            'image_path':      str(img_path),
            'annotation_path': ann_path,
            'batch_name':      batch_name,
        }
        try:
            # Try with the full modern signature first
            project.upload(
                **kwargs,
                tag_names     = ['sentinel2', 'ghana'],
                is_prediction = True,
            )
            uploaded += 1
        except TypeError:
            # Fall back for older roboflow versions
            try:
                project.upload(**kwargs)
                uploaded += 1
            except Exception as e2:
                msg = str(e2).lower()
                if 'already exists' in msg or 'duplicate' in msg:
                    skipped += 1
                else:
                    print(f'  ✗ {img_path.name}: {e2}')
                    errors += 1
        except Exception as e:
            msg = str(e).lower()
            if 'already exists' in msg or 'duplicate' in msg:
                skipped += 1
            else:
                print(f'  ✗ {img_path.name}: {e}')
                errors += 1

    print(f'\n  Uploaded : {uploaded}')
    print(f'  Skipped  : {skipped}  (already in project)')
    print(f'  Errors   : {errors}')


# ── Upload RGB images + pseudo-label annotations ─────────────────────────────
print('=== Uploading RGB images + pseudo-label annotations ===')
rgb_files = sorted(RGB_DIR.glob('*.jpg'))
print(f'Images to upload: {len(rgb_files)}')

if not rgb_files:
    print('⚠ No RGB exports found — run cell 10 first.')
else:
    ann_dir = PER_IMAGE_DIR if UPLOAD_ANNOTATIONS else None
    upload_batch(project, rgb_files, ann_dir=ann_dir)

# ── Optionally upload false-colour images as a second batch ──────────────────
if UPLOAD_FALSECOLOUR:
    print('\n=== Uploading false-colour images (reference batch) ===')
    fc_files = sorted(FC_DIR.glob('*.jpg'))
    upload_batch(project, fc_files, ann_dir=None,
                 batch_name=RF_BATCH_NAME + '-falsecolour')


## 11 — Verify upload

In [ ]:
# Pull a summary from the Roboflow API to confirm everything landed
try:
    stats = project.get_version_information()
    print('Project versions:')
    for v in stats:
        print(f'  v{v.get("version","?")} — {v.get("images","?")} images  '
              f'{v.get("annotations","?")} annotations')
except Exception:
    # Older roboflow package versions may not have get_version_information
    pass

print()
print('✓ Upload complete.')
print(f'  Open your project at: https://app.roboflow.com/{RF_WORKSPACE}/{RF_PROJECT}')
print()
print('Next steps in Roboflow:')
print('  1. Assign images to yourself (Annotate → Assign)')
print('  2. Review pre-drawn boxes — accept, correct, or redraw')
print('  3. Use false-colour NDVI images as visual reference (open in a second tab)')
print('  4. When done: Dataset → Generate → Export as COCO JSON')
print('  5. Download the export zip and update COCO_PATH in your training notebook')


## 12 — (After annotation) Download corrected labels from Roboflow

Run this cell **after** you have finished annotating in Roboflow.
It downloads your corrected COCO JSON export back to Kaggle working storage
so you can plug it straight into the training notebook.


In [ ]:
# ── Set these after your Roboflow annotation session ─────────────────────────
RF_EXPORT_VERSION = 1      # the dataset version number to download
RF_EXPORT_FORMAT  = 'coco' # 'coco' | 'yolov8' | 'pascal-voc'

DOWNLOAD_DIR = OUTPUT_DIR / 'roboflow_download'
DOWNLOAD_DIR.mkdir(exist_ok=True)

# Uncomment when ready to download:
# version  = project.version(RF_EXPORT_VERSION)
# dataset  = version.download(RF_EXPORT_FORMAT, location=str(DOWNLOAD_DIR))
# print(f'✓ Downloaded to {DOWNLOAD_DIR}')
# print()
# # The corrected COCO JSON will be at:
# train_json = DOWNLOAD_DIR / 'train' / '_annotations.coco.json'
# val_json   = DOWNLOAD_DIR / 'valid' / '_annotations.coco.json'
# print(f'  Train annotations : {train_json}')
# print(f'  Val   annotations : {val_json}')
# print()
# print('Update COCO_PATH in your training notebook to point to these files.')

print('Cell ready — uncomment the download block after annotating in Roboflow.')


## Appendix — Annotation tips for Sentinel-2 satellite imagery

### Visual cues to use in Roboflow

| What you see | Likely class | Confidence |
|---|---|---|
| Deep red/magenta in **false colour** | `forest` | High |
| Cyan / light blue-green in false colour | `degraded` / bare soil | High |
| Bright yellow-green in **NDVI** map | Transitional / young regrowth | Medium |
| Deep red/orange in NDVI | Bare earth, mining, exposed soil | High |
| Dark green in NDVI | Dense closed-canopy forest | High |
| Bright white in RGB | Cloud — **do not label** |  |
| Very dark/black in RGB | Cloud shadow — **do not label** |  |
| Rectangular bright patches | Agriculture / plantation — label if relevant |  |

### Recommended workflow per tile
1. Open the RGB image in Roboflow annotator
2. Open the false-colour image in a separate tab (same tile, `falsecolour/` batch)
3. Review the pre-drawn pseudo-label boxes — delete obvious cloud/shadow boxes
4. Add any large degraded areas the algorithm missed (especially open-cast mines)
5. Tighten box edges where the auto-label was coarse

### Threshold tuning
If you find the pseudo-labels are systematically wrong (too many / too few boxes),
edit these constants in **cell 2** and re-run cells 6 → 10:
- `NDVI_FOREST_THRESH` — raise to label only denser forest
- `NDVI_DEGRADED_THRESH` — lower to label only the most severely degraded areas
- `CLOUD_BRIGHTNESS_THRESH` — raise if too many land pixels are being masked out
